# CPL Cleaning Pipeline

Reads raw data from S3, cleans branches and census tracts, writes clean CSVs back to S3.
Defined as a KFP pipeline so runs are traceable in the RHOAI dashboard.

In [ ]:
!pip install -q kfp==2.* boto3 pandas geopandas shapely pyproj

In [ ]:
import os
import kfp
from kfp import dsl
from kfp.dsl import Input, Output, Dataset

## Pipeline Components

In [ ]:
@dsl.component(
    base_image="quay.io/modh/runtime-images:runtime-cuda-tensorflow-ubi9-python-3.9",
    packages_to_install=["boto3", "pandas"],
)
def clean_branches(
    s3_endpoint: str,
    s3_bucket: str,
    aws_access_key: str,
    aws_secret_key: str,
    branches_csv: Output[Dataset],
):
    import ast
    import io
    import boto3
    import pandas as pd

    s3 = boto3.client(
        "s3",
        endpoint_url=s3_endpoint,
        aws_access_key_id=aws_access_key,
        aws_secret_access_key=aws_secret_key,
        verify=False,
    )

    raw_key = "raw/branches/Libraries_-_Locations___Contact_Information__and_Usual_Hours_of_Operation.csv"
    obj = s3.get_object(Bucket=s3_bucket, Key=raw_key)
    df = pd.read_csv(io.BytesIO(obj["Body"].read()))

    df.columns = df.columns.str.strip()
    df = df.rename(columns={"NAME": "BRANCH"})

    name_fixes = {
        "Daley, Richard J.": "Daley, Richard J. - Bridgeport",
        "Daley, Richard M.": "Daley, Richard M. - W. Humboldt",
        "Harold Washtington Library Center": "Harold Washington Library Center",
        "King": "Martin Luther King Jr.",
    }
    df["BRANCH"] = df["BRANCH"].replace(name_fixes)
    df["PERMANENTLY_CLOSED"] = 0

    roosevelt = pd.DataFrame([{
        "BRANCH": "Roosevelt",
        "HOURS OF OPERATION": None,
        "ADDRESS": "1101 W. Taylor Street",
        "CITY": "Chicago", "STATE": "IL", "ZIP": "60607",
        "PHONE": None, "WEBSITE": None,
        "LOCATION": "(41.86943838326072, -87.6544586134924)",
        "PERMANENTLY_CLOSED": 1,
    }])
    df = pd.concat([df, roosevelt], ignore_index=True)

    coords = df["LOCATION"].apply(ast.literal_eval)
    df["LATITUDE"] = coords.apply(lambda c: float(c[0]))
    df["LONGITUDE"] = coords.apply(lambda c: float(c[1]))

    df = df.sort_values("BRANCH").reset_index(drop=True)

    # Write to pipeline artifact
    df.to_csv(branches_csv.path, index=False)

    # Also write to S3
    buf = io.BytesIO()
    df.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=s3_bucket, Key="clean/branches.csv", Body=buf.getvalue())
    print(f"Wrote {len(df)} branches to s3://{s3_bucket}/clean/branches.csv")

In [ ]:
@dsl.component(
    base_image="quay.io/modh/runtime-images:runtime-cuda-tensorflow-ubi9-python-3.9",
    packages_to_install=["boto3", "pandas", "geopandas", "shapely", "pyproj", "pyogrio"],
)
def clean_census_tracts(
    s3_endpoint: str,
    s3_bucket: str,
    aws_access_key: str,
    aws_secret_key: str,
    tracts_csv: Output[Dataset],
):
    import io
    import tempfile
    import zipfile
    import boto3
    import pandas as pd
    import geopandas as gpd

    s3 = boto3.client(
        "s3",
        endpoint_url=s3_endpoint,
        aws_access_key_id=aws_access_key,
        aws_secret_access_key=aws_secret_key,
        verify=False,
    )

    # Load Chicago city boundary
    obj = s3.get_object(Bucket=s3_bucket, Key="raw/boundaries/Boundaries - City.geojson")
    chicago = gpd.read_file(io.BytesIO(obj["Body"].read()))

    # Load Illinois census tracts (zip file)
    obj = s3.get_object(Bucket=s3_bucket, Key="raw/boundaries/tl_2021_17_tract.zip")
    with tempfile.NamedTemporaryFile(suffix=".zip") as tmp:
        tmp.write(obj["Body"].read())
        tmp.flush()
        il_tracts = gpd.read_file(tmp.name)

    il_tracts = il_tracts.to_crs(chicago.crs)
    chicago_tracts = gpd.clip(il_tracts, chicago)

    # Load demographics
    obj = s3.get_object(Bucket=s3_bucket, Key="raw/demographics/R13395758_SL140.csv")
    demos = pd.read_csv(io.BytesIO(obj["Body"].read()))
    demos = demos.iloc[1:, :]

    merged = chicago_tracts.merge(demos, how="left", left_on="GEOID", right_on="FIPS")

    id_cols = {
        "GEOID": "geoid",
        "Qualifying Name": "qualifying name",
        "ALAND": "current land area (sq. m)",
        "AWATER": "current water area (sq. m)",
    }
    merged = merged.rename(columns=id_cols)

    pop_cols = [c for c in merged.columns if c.startswith("Total Population") or
                c.startswith("Population Density") or c.startswith("Total:")]

    keep = list(id_cols.values()) + pop_cols + ["geometry"]
    keep = [c for c in keep if c in merged.columns]
    result = merged[keep].copy()
    result.columns = result.columns.str.lower()

    # Write to pipeline artifact
    result.to_csv(tracts_csv.path, index=False)

    # Write to S3
    buf = io.BytesIO()
    result.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=s3_bucket, Key="clean/population_demographics.csv", Body=buf.getvalue())
    print(f"Wrote {len(result)} tracts to s3://{s3_bucket}/clean/population_demographics.csv")

## Pipeline Definition

In [ ]:
@dsl.pipeline(
    name="cpl-cleaning",
    description="Clean raw CPL branch and census tract data for coverage analysis",
)
def cleaning_pipeline(
    s3_endpoint: str = "https://s3.openshift-storage.svc:443",
    s3_bucket: str = "cpl-coverage-data",
    aws_access_key: str = "",
    aws_secret_key: str = "",
):
    clean_branches(
        s3_endpoint=s3_endpoint,
        s3_bucket=s3_bucket,
        aws_access_key=aws_access_key,
        aws_secret_key=aws_secret_key,
    )
    clean_census_tracts(
        s3_endpoint=s3_endpoint,
        s3_bucket=s3_bucket,
        aws_access_key=aws_access_key,
        aws_secret_key=aws_secret_key,
    )

## Compile & Submit

In [ ]:
from kfp.compiler import Compiler

Compiler().compile(cleaning_pipeline, "cleaning_pipeline.yaml")
print("Pipeline compiled to cleaning_pipeline.yaml")

In [ ]:
# Submit to the DSPA running in this namespace
# The route is auto-discovered from the DSPA service
import subprocess

namespace = subprocess.check_output(
    ["cat", "/var/run/secrets/kubernetes.io/serviceaccount/namespace"]
).decode().strip()

client = kfp.Client(
    host=f"https://ds-pipeline-dspa.{namespace}.svc:8443",
    verify_ssl=False,
)

run = client.create_run_from_pipeline_package(
    "cleaning_pipeline.yaml",
    arguments={
        "s3_endpoint": os.environ.get("AWS_S3_ENDPOINT", "https://s3.openshift-storage.svc:443"),
        "s3_bucket": os.environ.get("AWS_S3_BUCKET", "cpl-coverage-data"),
        "aws_access_key": os.environ.get("AWS_ACCESS_KEY_ID", ""),
        "aws_secret_key": os.environ.get("AWS_SECRET_ACCESS_KEY", ""),
    },
    run_name="cpl-cleaning-run",
)
print(f"Pipeline run submitted: {run.run_id}")